# 6장 · HTTP 요청과 JSON 응답 처리

HTTP 요청에서는 경로가 대상을 지정하고 상태 코드가 처리 결과를 알려 줍니다. 성공 응답을 받아도 본문이 예상한 JSON인지 따로 확인해야 합니다.

할 일 1번을 조회하는 예제로 요청 경로, HTTP 상태, JSON 본문, 응답 ID를 검사합니다. 기본 실습은 가상 응답을 사용하며 외부 계정 없이 실행할 수 있습니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 httpx 0.28.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 경로 매개변수와 요청 ID

GET은 값을 읽어 달라는 요청입니다. /todos/1에서 /todos는 자료 종류, 마지막 1은 요청한 기록 번호입니다. 서버는 상태 코드와 본문을 돌려줍니다.


### 선택 실습 · 공개 API 요청

<p>timeout=5.0은 연결·읽기 등 네트워크 단계의 대기 제한에 사용됩니다. 전체 프로그램이 반드시 5초 안에 끝난다는 보장은 아닙니다. 무제한 대기를 피하려는 설정이며 상세 시간 측정은 별도로 해야 합니다.</p>


### 셀 01 · f-string으로 경로 구성

입력받은 번호를 주소에서 한 번만 사용하도록 만듭니다.


In [1]:
todo_id = 1
path = f"/todos/{todo_id}"
print(path)

/todos/1


/todos/1입니다. 뒤에서 받은 데이터의 id가 1인지 다시 확인할 때 같은 todo_id를 사용합니다.


### HTTP 리소스와 응답

<p>HTTP 요청은 서버에 어떤 자원을 어떻게 다룰지 전달하는 메시지입니다. URL은 주소, GET은 조회 방법입니다. 서버는 상태 코드·헤더·본문을 응답합니다. JSON은 응답 본문에 담길 수 있는 데이터 형식이며 HTTP 자체와 같은 뜻은 아닙니다.</p>

<p>기본 실습에서는 /todos/1이라는 요청 경로를 만들고 미리 적어 둔 JSON 본문을 읽습니다. 셀 05에서 HTTPX가 받는 404도 수업용 함수가 만든 응답입니다. 실제 서버에 접속하는 경우는 공개 API 선택 실습으로 구분합니다.</p>

<table><caption>URL·응답 구성</caption><thead><tr><th scope="col">부분</th><th scope="col">예</th><th scope="col">역할</th></tr>
</thead><tbody><tr><td>주소</td><td>https://jsonplaceholder.typicode.com</td><td>연결할 서비스</td></tr>
<tr><td>경로</td><td>/todos/1</td><td>조회할 자원</td></tr>
<tr><td>쿼리</td><td>?userId=1</td><td>조회 조건의 예</td></tr>
<tr><td>상태 코드</td><td>200 / 404</td><td>처리 결과의 종류</td></tr>
<tr><td>헤더</td><td>Content-Type: application/json</td><td>본문 형식 등의 정보</td></tr>
<tr><td>본문</td><td>id·title·completed</td><td>실제 데이터</td></tr>
</tbody></table>

<table><caption>이 장에서 구분할 상태</caption><thead><tr><th scope="col">범위·예</th><th scope="col">뜻</th><th scope="col">대응 예</th></tr>
</thead><tbody><tr><td>2xx / 200</td><td>서버가 성공으로 응답</td><td>본문 형식도 추가 확인</td></tr>
<tr><td>400</td><td>잘못된 요청</td><td>요청 값·문서 확인</td></tr>
<tr><td>401 / 403</td><td>인증 필요 / 접근 거부</td><td>권한·인증 설정 확인</td></tr>
<tr><td>404</td><td>해당 자원 없음</td><td>경로·식별자 확인</td></tr>
<tr><td>429</td><td>요청 제한</td><td>즉시 반복하지 않고 정책 확인</td></tr>
<tr><td>5xx</td><td>서버 측 실패</td><td>기록하고 재시도 정책 확인</td></tr>
</tbody></table>

<table><caption>조회 실패를 구분하는 근거</caption><thead><tr><th scope="col">관찰한 값</th><th scope="col">검사 위치</th><th scope="col">다음에 확인할 내용</th></tr></thead><tbody><tr><td>/todos/1</td><td>요청 경로</td><td>조회한 번호가 1인지</td></tr><tr><td>404</td><td>HTTP 상태</td><td>응답은 도착했지만 자원을 찾지 못함</td></tr><tr><td>JSONDecodeError</td><td>본문 해석</td><td>받은 글이 JSON인지</td></tr><tr><td>id가 2</td><td>응답 내용</td><td>요청한 1번 기록과 다른 데이터</td></tr></tbody></table>

### 셀 02 · JSON 응답 디코딩

서버가 보냈다고 가정한 JSON 한 건을 Python 값으로 바꿉니다.


In [2]:
import json
body = '{"id":1,"title":"  HTTP 연습  ","completed":false}'
todo = json.loads(body)
print(todo["title"].strip())

HTTP 연습


디코딩한 title 값은 ‘HTTP 연습’입니다.

요청 주소는 1번인데 응답의 id만 2로 바꿔 보세요. JSON을 읽는 줄은 여전히 성공합니다. 잘못된 할 일을 화면에 띄우지 않으려면 해석한 객체의 id와 처음 주소를 만들 때 쓴 todo_id를 대조해야 합니다.


#### 요청 성공과 JSON 읽기

<mark>상태 코드와 응답 본문은 따로 확인합니다.</mark> 서버가 응답을 보냈어도 본문이 JSON이라는 보장은 없습니다. 상태 검사에서 실패했는지, 본문을 해석하다 실패했는지 나누면 다시 확인할 대상이 좁아집니다.

선수 개념 · 3장 함수의 반환값·타입 힌트·예외 처리

### 과제 03 · 요청 ID와 응답 ID 대조

응답에서 id만 2로 바꿨습니다. 받은 번호가 요청 번호와 같은지 출력하세요.


In [3]:
received = {**todo, "id": 2}
# 받은 번호와 요청한 번호를 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>received = {**todo, &quot;id&quot;: 2}
print(received[&quot;id&quot;] == todo_id)</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>False이므로 이 제목을 1번 기록으로 표시하면 안 됩니다. 상태 코드가 200인 경우에도 검사할 수 있는 내용입니다.</p>
</details>


### 응답 ID가 원래 요청과 같은지 확인

<p>이 실습은 1번을 요청했지만 received에는 id가 2인 응답을 넣습니다. 응답 본문을 읽을 수 있어도 다른 번호의 자료일 수 있으므로 받은 id와 todo_id를 직접 대조하세요.</p>


## HTTP 상태 오류와 JSONDecodeError

404는 요청에 응답했지만 해당 자원을 찾지 못했다는 상태입니다. 타임아웃이면 읽을 응답 본문조차 없을 수 있습니다. 상태 확인을 먼저 하고 성공했을 때만 JSON을 읽습니다.


### 모의 응답별 분기 테스트

<p>셀 05의 missing은 어떤 경로를 받아도 상태 코드 404를 반환합니다. 주소만 바꾸어서는 성공 응답이 나오지 않습니다. 응답을 만드는 함수가 무엇을 돌려주는지와 클라이언트가 그 상태를 어떻게 처리하는지를 따로 읽으세요.</p>

<p>응답이 도착했기 때문에 이 예외에는 error.response가 있습니다. 서버에서 아무 응답도 받지 못한 시간 초과라면 이곳에서 404를 읽을 수 없습니다.</p>

### 셀 04 · JSONDecodeError 처리

상태 코드는 200인데 본문에는 HTML이 담겼다고 가정합니다. 어떤 변환에서 실패하는지 확인하세요.


In [4]:
try:
    json.loads("<html>점검 중</html>")
except json.JSONDecodeError as error:
    print(type(error).__name__)

JSONDecodeError


JSONDecodeError가 나옵니다. URL을 다시 요청하기 전에 이미 받은 본문의 종류와 형식을 점검할 수 있습니다.


### HTTP 상태와 JSON 디코딩 따로 확인하기

<p>서버 상태를 확인하는 raise_for_status()와 본문을 해석하는 json.loads()는 실패 지점이 다릅니다. 점검용 HTML처럼 JSON이 아닌 본문은 JSONDecodeError 경로에서 확인합니다.</p>


### 셀 05 · HTTPX의 상태 코드 예외

httpx가 설치된 수업 커널에서는 MockTransport가 정한 상태로 응답합니다. 네트워크 연결은 없습니다.

client.get(path)를 호출하면 MockTransport가 이 셀의 missing(request)를 대신 부릅니다. Response(404)를 만든 곳은 바로 그 함수입니다. 인터넷을 끊어도 이 404를 관찰할 수 있습니다.


In [5]:
from importlib.util import find_spec
if find_spec("httpx") is not None:
    import httpx
    def missing(request):
        return httpx.Response(404, json={"detail": "할 일을 찾을 수 없습니다."})
    with httpx.Client(transport=httpx.MockTransport(missing), base_url="https://example.invalid") as client:
        response = client.get(path)
        try:
            response.raise_for_status()
        except httpx.HTTPStatusError as error:
            print(error.response.status_code)
else:
    print("HTTPX가 설치된 커널에서 404 응답 실습을 실행하세요.")

404


HTTPX를 사용한 출력은 404입니다. 연결 실패 예외와 달리 error.response에서 서버의 상태 값을 읽습니다. 외부 서버의 가동 여부는 이 모의 요청으로 알 수 없습니다.


### MockTransport가 돌려준 HTTP 응답

<p>404가 나온 요청에서는 본문을 해석하기 전에 raise_for_status()가 흐름을 멈춥니다.</p>


## JSON 불리언과 문자열 구분

JSON의 false는 파이썬의 False가 되지만 따옴표가 있는 "false"는 비어 있지 않은 문자열입니다. 자료형을 먼저 검사하면 잘못된 완료 표시를 막을 수 있습니다.


### 과제 06 · bool 타입 검사

잘못 들어온 문자열을 bool()로 바꾸지 말고 완료 값이 실제 bool인지 확인하세요.


In [6]:
completed = "false"
# completed의 자료형을 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>completed = &quot;false&quot;
print(type(completed) is bool)</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>False가 나옵니다. 응답 형식 오류로 기록할 입력입니다. bool(completed)를 쓰면 내용과 관계없이 비어 있지 않은 문자열이어서 True가 됩니다.</p>
</details>


### JSON 구조와 필드 검증

<p>과제 03은 요청과 응답의 id가 일치하는지, 과제 06은 completed가 파이썬 bool인지 검사합니다. 두 조건은 독립적입니다. 번호가 일치해도 completed가 문자열이면 완료 상태로 사용할 수 없습니다.</p>

<p>isinstance(값, 종류)는 해당 종류인지 검사합니다. bool은 Python에서 int의 하위 종류이므로 id에 True가 들어오는 것을 막으려면 type(todo_id) is int처럼 정확한 종류를 확인할 수 있습니다. 여기서는 숫자와 bool을 구분하는 과제 규칙에 사용합니다.</p>

<p>문자열 "false"를 bool(...)로 바꾸면 비어 있지 않아 True가 됩니다. completed를 임의 변환하지 말고 JSON의 true/false가 Python bool로 읽혔는지 확인합니다.</p>

## 페이지가 둘로 나뉜 할 일 목록 조회하기

GET 한 번으로 목록의 일부만 받는 API를 다룹니다. 첫 응답의 next 값이 있으면 다음 페이지의 주소가 됩니다. HTTPX의 실제 Client로 요청을 보내되, 예제 서버의 응답은 MockTransport에서 제공합니다.

### 셀 09 · 첫 페이지의 본문 읽기

수업용 서버는 첫 페이지에 1번 할 일과 다음 주소를 넣습니다. 두 번째 요청 전에는 2번 할 일이 보이지 않습니다.

In [7]:
import httpx
todo_pages = {
    "/todos": {"items": [{"id": 1, "title": "HTTP 연습"}], "next": "/todos/next"},
    "/todos/next": {"items": [{"id": 2, "title": "응답 확인"}], "next": None},
}
def serve_page(request):
    page = todo_pages.get(request.url.path)
    if page is None:
        return httpx.Response(404, json={"detail": "페이지 없음"})
    return httpx.Response(200, json=page)

with httpx.Client(transport=httpx.MockTransport(serve_page), base_url="https://example.invalid") as client:
    first_page = client.get("/todos")
    first_page.raise_for_status()
    page_data = first_page.json()
print(page_data["items"][0]["title"], page_data["next"])

HTTP 연습 /todos/next


HTTP 연습 옆의 /todos/next는 다음 요청에 쓸 경로입니다. 화면에 표시할 두 번째 할 일의 제목은 이 응답에 들어 있지 않습니다.

### 과제 10 · 서버가 준 다음 주소를 따라가기

같은 모의 서버에서 page_data["next"]를 요청하세요. 상태를 검사한 다음 첫 페이지와 두 번째 페이지의 items를 이어 붙여 ID 목록과 마지막 next를 출력하세요.

In [8]:
next_path = page_data["next"]
# 다음 페이지를 요청하고 두 페이지의 ID를 합치세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>next_path = page_data[&quot;next&quot;]
with httpx.Client(transport=httpx.MockTransport(serve_page), base_url=&quot;https://example.invalid&quot;) as client:
    next_response = client.get(next_path)
    next_response.raise_for_status()
    last_page = next_response.json()
items = page_data[&quot;items&quot;] + last_page[&quot;items&quot;]
print([item[&quot;id&quot;] for item in items], last_page[&quot;next&quot;])</code></pre>
<p>실행 결과</p><pre><code>[1, 2] None</code></pre>
<p>[1, 2]에서 두 요청의 자료가 합쳐진 것을 확인할 수 있습니다. 마지막 None은 서버가 다음 주소를 주지 않았다는 뜻입니다. 주소를 임의로 /todos/2라고 지으면 이 서버는 404를 돌려줍니다.</p>
</details>

<mark>첫 요청이 성공해도 목록 전체를 받았는지는 다음 페이지 표시를 읽어야 압니다.</mark> 다음 주소를 조합해서 추측하지 말고 응답에서 받은 경로를 사용하세요. 마지막 페이지에서 next가 None이면 여기서 요청을 마칩니다.

응답 본문을 읽기 전에 상태를 확인하는 순서는 페이지가 늘어나도 같습니다.

## 돌아보기

주소를 잘못 썼을 때, 기다리는 동안 연결이 끊겼을 때, JSON의 칸이 잘못됐을 때를 차례로 떠올려 보세요. 같은 ‘조회 실패’ 화면에서도 기록할 정보와 다음에 확인할 곳이 달라집니다.

### API 호출 결과와 운영 기록

<table><caption>실패 기록에 남길 내용</caption><thead><tr><th scope="col">항목</th><th scope="col">예</th></tr>
</thead><tbody><tr><td>실행 종류</td><td>모의 / 실제</td></tr>
<tr><td>요청</td><td>GET /todos/1, 공개 연습 데이터</td></tr>
<tr><td>결과</td><td>성공 / 상태 코드 / 형식 오류 / 연결 실패</td></tr>
<tr><td>확인 범위</td><td>JSON 검증까지 / 외부 연결 미확인</td></tr>
<tr><td>다음 행동</td><td>입력 수정·네트워크 확인·수업용 모의 전환</td></tr>
</tbody></table>
